<a href="https://colab.research.google.com/github/shris2810/Langchain-Langgraph/blob/main/simple_AIAgent_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Hugging Face access token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

In [ ]:
!pip install -q langchain-openai langchain-community langchain-core requests duckduckgo-search

In [ ]:
pip install -U langchain-huggingface sentence-transformers

In [ ]:
pip install -U langchain-community ddgs

In [ ]:
pip install -U langgraph langchain-openai langchain-core

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
search_tool = DuckDuckGoSearchRun()

In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)

model = ChatHuggingFace(
    llm=llm
)

In [ ]:
from langchain_core.prompts import PromptTemplate

react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_template)

In [ ]:
from langchain.agents import create_agent

tools = [search_tool, get_weather_data]

# Create the agent
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools step-by-step to gather information and answer questions."
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the capital of Madhya Pradesh, then find its current weather condition"}
    ]
})

print(response["messages"][-1].content)

# older version

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent

# 4. Create classic agent
agent = create_react_agent(
    llm=model,
    tools=tools,
    prompt=prompt
)

# 5. Wrap in AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True  # for open-source models
)

response = agent_executor.invoke({
    "input": "Find the capital of Madhya Pradesh, then find its current weather condition"
})
print(response["output"])